## The goal
Build a single 779-dim user feature vector from your personal Goodreads CSV:

`[ like_emb (384) | dislike_emb (384) | genre_dist (10) | mean_pages (1) ]`

Built entirely from artifacts produced earlier:
- `book_embeddings.npy` (step 11) — 384-dim per book
- `genre_matrix.npy` (step 12) — 10-dim L2-normalized per book
- `num_pages_normalized.npy` (step 12) — scalar in [0, 1] per book
- `book_id_to_index.json` — UCSD `book_id` -> row index in the three matrices above
- `my_books.csv` — your `(book_id, my_rating)` rows after filtering to `my_rating > 0`

Rules:
- 4+ stars = liked, 1-2 stars = disliked, 3 stars = excluded.
- Like embedding is **rating-weighted** (5-star counts more than 4-star).
- Dislike embedding is a **simple mean**.
- Genre distribution and mean pages come from **liked books only**.

In [1]:
import json
import numpy as np
import polars as pl
from mybookrec import DATA_DIR

# Load every artifact this step depends on. The three numpy matrices are aligned by row
# index, and `book_id_to_index` is the lookup that ties a UCSD book_id to that row.
transformed = DATA_DIR / "transformed"

my_books = pl.read_csv(transformed / "my_books.csv")
with open(transformed / "book_id_to_index.json", "r") as f:
    book_id_to_index = json.load(f)  # keys are string book_ids, values are int row indices

book_embeddings = np.load(transformed / "book_embeddings.npy")     # (n_books, 384)
genre_matrix = np.load(transformed / "genre_matrix.npy")           # (n_books, 10)
pages_vec = np.load(transformed / "num_pages_normalized.npy")       # (n_books,)

print(f"my_books rows:        {len(my_books):,}")
print(f"book_id_to_index:     {len(book_id_to_index):,} entries")
print(f"book_embeddings:      {book_embeddings.shape} dtype={book_embeddings.dtype}")
print(f"genre_matrix:         {genre_matrix.shape}")
print(f"pages_vec:            {pages_vec.shape}")

my_books rows:        293
book_id_to_index:     1,782,579 entries
book_embeddings:      (1782579, 384) dtype=float16
genre_matrix:         (1782579, 10)
pages_vec:            (1782579,)


In [2]:
# Match each of my books to its UCSD row index. Books not in the UCSD corpus get dropped:
# typically very recent releases that postdate the dataset snapshot.
# JSON keys are strings, my_books.book_id is Int64, so cast before lookup.
matched = (
    my_books
    .with_columns(pl.col("book_id").cast(pl.String).alias("book_id_str"))
    .with_columns(
        pl.col("book_id_str").map_elements(
            lambda b: book_id_to_index.get(b, -1), return_dtype=pl.Int64 # -1 means "not found"
        ).alias("row_idx")
    )
)

n_input = len(matched)
dropped = matched.filter(pl.col("row_idx") == -1)
matched = matched.filter(pl.col("row_idx") != -1)

print(f"My books in CSV:        {n_input}")
print(f"Matched to UCSD:        {len(matched)}")
print(f"Dropped (not in UCSD):  {len(dropped)}")
if len(dropped) > 0:
    print("\nDropped book_ids:", dropped["book_id"].to_list())

My books in CSV:        293
Matched to UCSD:        293
Dropped (not in UCSD):  0


In [3]:
# Split into liked (4+) and disliked (1-2). 3-star books are excluded — ambiguous taste signal.
liked = matched.filter(pl.col("my_rating") >= 4)
disliked = matched.filter(pl.col("my_rating") <= 2)
excluded = matched.filter(pl.col("my_rating") == 3)

liked_idx = liked["row_idx"].to_numpy()
liked_ratings = liked["my_rating"].to_numpy().astype(np.float32)
disliked_idx = disliked["row_idx"].to_numpy()

print(f"Liked (4+):     {len(liked_idx)}")
print(f"Disliked (1-2): {len(disliked_idx)}")
print(f"Excluded (3):   {len(excluded)}")

Liked (4+):     243
Disliked (1-2): 42
Excluded (3):   8


In [4]:
# Like embedding: rating-weighted mean of embeddings for 4+ star books.
# weighted_mean = sum(rating_i * embedding_i) / sum(rating_i)
# Higher ratings pull the centroid toward the books you liked most.
if len(liked_idx) == 0:
    raise ValueError("No liked books matched UCSD — cannot build like embedding.")

liked_embeddings = book_embeddings[liked_idx]                # (n_liked, 384)
weights = liked_ratings[:, None]                              # (n_liked, 1) — column for broadcasting
like_emb = (liked_embeddings * weights).sum(axis=0) / weights.sum()
like_emb = like_emb.astype(np.float32)

print(f"like_emb shape: {like_emb.shape}, L2 norm: {np.linalg.norm(like_emb):.4f}")

like_emb shape: (384,), L2 norm: 0.7818


In [5]:
# Dislike embedding: simple mean of embeddings for 1-2 star books.
# If you have no disliked books, fall back to a zero vector — the model will learn to ignore it.
if len(disliked_idx) == 0:
    print("WARNING: no disliked books matched UCSD. Using zero vector for dislike_emb.")
    dislike_emb = np.zeros(book_embeddings.shape[1], dtype=np.float32)
else:
    dislike_emb = book_embeddings[disliked_idx].mean(axis=0).astype(np.float32)

print(f"dislike_emb shape: {dislike_emb.shape}, L2 norm: {np.linalg.norm(dislike_emb):.4f}")
print(f"  (built from {len(disliked_idx)} disliked books — small samples mean noisy signal)")

dislike_emb shape: (384,), L2 norm: 0.7888
  (built from 42 disliked books — small samples mean noisy signal)


In [6]:
# Genre distribution: sum the genre vectors of liked books, then L2-normalize.
# This produces a unit-length "shape of taste" vector in the same 10-dim space as item genres.
liked_genres = genre_matrix[liked_idx]                        # (n_liked, 10) — already L2'd per book
genre_dist = liked_genres.sum(axis=0)
norm = np.linalg.norm(genre_dist)
genre_dist = (genre_dist / norm if norm > 0 else genre_dist).astype(np.float32)

# Mean pages: average of normalized page values across liked books.
# Already in [0, 1] from item features, so the mean is too.
mean_pages = float(pages_vec[liked_idx].mean())

print(f"genre_dist shape: {genre_dist.shape}, L2 norm: {np.linalg.norm(genre_dist):.4f}")
print(f"genre_dist values: {genre_dist}")
print(f"mean_pages: {mean_pages:.4f}")

genre_dist shape: (10,), L2 norm: 1.0000
genre_dist values: [0.644207   0.33088395 0.08759174 0.02514423 0.10236984 0.21617489
 0.063712   0.08690079 0.4535761  0.4389251 ]
mean_pages: 0.3518


In [ ]:
# Concatenate the four pieces into the final user feature vector.
# Order matters: it must match the order the UserTower expects at training/inference. so first 384 dims as like embedding, then 384 for dislike, then 10 for genre distribution, then 1 for mean pages.
user_features = np.concatenate([
    like_emb,                       # 384
    dislike_emb,                    # 384 
    genre_dist,                     # 10
    np.array([mean_pages], dtype=np.float32),  # 1
]).astype(np.float32)

expected_dim = book_embeddings.shape[1] * 2 + genre_matrix.shape[1] + 1
assert user_features.shape == (expected_dim,), f"Expected ({expected_dim},), got {user_features.shape}"
assert not np.isnan(user_features).any(), "NaNs in user_features — check upstream"

np.save(transformed / "user_features.npy", user_features)
print(f"Saved user_features of shape {user_features.shape} to data/transformed/user_features.npy")

Saved user_features of shape (779,) to data/transformed/user_features.npy
